# 11 · Working with Text Data (NLP basics)

Text isn't numbers, so before a model can touch it we must **vectorize** it. This notebook
covers the classic, still-everywhere pipeline:

- **Bag-of-words** counts and **TF-IDF** weighting.
- A full **text-classification** pipeline (vectorizer + classifier).
- Peeking at **which words drive predictions**.
- **Topic modeling** to discover themes without labels.

We use a small built-in newsgroup subset so everything runs fast.

In [1]:
%matplotlib inline
import numpy as np, matplotlib.pyplot as plt

## 1. Bag-of-words: text → a count matrix

`CountVectorizer` builds a vocabulary and represents each document as counts of each word.
Order is thrown away ("bag"), but for many tasks word *presence* is enough.

In [2]:
from sklearn.feature_extraction.text import CountVectorizer

docs = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "cats and dogs are great pets",
]
cv = CountVectorizer()
counts = cv.fit_transform(docs)
print("vocabulary:", cv.get_feature_names_out())
print("count matrix (docs × words):\n", counts.toarray())

vocabulary: ['and' 'are' 'cat' 'cats' 'dog' 'dogs' 'great' 'log' 'mat' 'on' 'pets'
 'sat' 'the']
count matrix (docs × words):
 [[0 0 1 0 0 0 0 0 1 1 0 1 2]
 [0 0 0 0 1 0 0 1 0 1 0 1 2]
 [1 1 0 1 0 1 1 0 0 0 1 0 0]]


## 2. TF-IDF: down-weight boring words

Common words ("the", "on") appear everywhere and carry little signal. **TF-IDF** multiplies a
word's frequency in a document by how *rare* it is across all documents, so distinctive words
score higher. It's usually a better default than raw counts.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()
mat = tfidf.fit_transform(docs)
import pandas as pd
show = pd.DataFrame(mat.toarray().round(2), columns=tfidf.get_feature_names_out())
show

,and,are,cat,cats,dog,dogs,great,log,mat,on,pets,sat,the
0,0.00,0.00,0.43,0.00,0.00,0.00,0.00,0.00,0.43,0.33,0.00,0.33,0.65
1,0.00,0.00,0.00,0.00,0.43,0.00,0.00,0.43,0.00,0.33,0.00,0.33,0.65
2,0.41,0.41,0.00,0.41,0.00,0.41,0.41,0.00,0.00,0.00,0.41,0.00,0.00


## 3. A text classifier

We classify short documents from two very different themes — **hockey** vs **space**. To keep
this notebook fully offline, we synthesize a small corpus by sampling from theme-specific
vocabularies. The pipeline is just TF-IDF followed by logistic regression — a strong, fast
baseline that's hard to beat for many text tasks.

In [4]:
# Build a small labeled corpus with no network needed.
rng = np.random.RandomState(0)

hockey = "hockey puck goal skate ice rink team player stick penalty coach score shot".split()
space  = "space rocket orbit planet galaxy nasa launch satellite mission star gravity moon".split()
filler = "the a of to and in on with for is are this that it was".split()

def make_doc(topic_words, length=25):
    words = list(rng.choice(topic_words, size=length//2)) + list(rng.choice(filler, size=length//2))
    rng.shuffle(words)
    return " ".join(words)

docs_train = [make_doc(hockey) for _ in range(150)] + [make_doc(space) for _ in range(150)]
y_train    = [0]*150 + [1]*150
docs_test  = [make_doc(hockey) for _ in range(50)]  + [make_doc(space) for _ in range(50)]
y_test     = [0]*50 + [1]*50
target_names = ["hockey", "space"]

from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

clf = make_pipeline(TfidfVectorizer(stop_words="english", min_df=3),
                    LogisticRegression(max_iter=2000))
clf.fit(docs_train, y_train)
print(f"test accuracy: {clf.score(docs_test, y_test):.1%}")

test accuracy: 100.0%


In [5]:
# Which words push toward each class? Inspect the largest positive/negative weights.
vec = clf.named_steps["tfidfvectorizer"]
lr  = clf.named_steps["logisticregression"]
words = np.array(vec.get_feature_names_out())
coefs = lr.coef_[0]
order = np.argsort(coefs)

print(f"most '{target_names[0]}' words:", list(words[order[:8]]))
print(f"most '{target_names[1]}' words:", list(words[order[-8:]]))

most 'hockey' words: ['puck', 'stick', 'ice', 'player', 'skate', 'team', 'rink', 'score']
most 'space' words: ['planet', 'space', 'rocket', 'galaxy', 'nasa', 'satellite', 'orbit', 'star']


## 4. Topic modeling: themes without labels

**LDA (Latent Dirichlet Allocation)** is unsupervised: it discovers clusters of co-occurring
words ("topics") across documents, with no labels at all. Useful for exploring a large corpus
you haven't read. We feed it a mixed corpus of three themes and check that it recovers them.

In [6]:
from sklearn.decomposition import LatentDirichletAllocation

food = "pizza pasta recipe kitchen chef flavor dish cook oven spice meal taste".split()
corpus = ([make_doc(hockey) for _ in range(120)] +
          [make_doc(space)  for _ in range(120)] +
          [make_doc(food)   for _ in range(120)])

cv = CountVectorizer(max_df=0.9, min_df=3, stop_words="english")
X = cv.fit_transform(corpus)
lda = LatentDirichletAllocation(n_components=3, random_state=0, learning_method="batch").fit(X)

vocab = cv.get_feature_names_out()
for t, comp in enumerate(lda.components_):
    top = vocab[np.argsort(comp)[-8:]][::-1]
    print(f"topic {t}: {', '.join(top)}")

topic 0: dish, meal, flavor, cook, chef, taste, oven, kitchen
topic 1: ice, goal, hockey, penalty, shot, rink, score, player
topic 2: nasa, star, gravity, galaxy, orbit, space, planet, rocket


## Recap

- Vectorize text with **bag-of-words** counts or, better, **TF-IDF** weighting.
- **TF-IDF + logistic regression** is a strong, fast text-classification baseline.
- Inspecting model **coefficients** reveals the words driving each class.
- **LDA** discovers latent **topics** with no labels — great for corpus exploration.

**Next:** `12 — MLOps`, where a trained pipeline becomes a served, tracked, containerized service.